# QueueMind — self-hosted Llama fast lane (Colab)

Runs a small Llama model on Colab's free GPU and plugs it into QueueMind's model router.

**What it does**
1. Starts [Ollama](https://ollama.com) with a Llama model on the T4 GPU.
2. Opens a free Cloudflare quick tunnel so the deployed app can reach it over HTTPS.
3. Announces the tunnel URL to the app (`/api/runtime`) and keeps sending heartbeats every 30 s.

While this notebook runs, QueueMind's fast lane (intake triage, SMS drafting) prefers this model — patient text stays on a model *we* host.
When the notebook stops, heartbeats stop, and within ~90 s the router falls back to Groq (gpt-oss-20b) → Gemini automatically.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and add two Colab secrets (🔑 icon on the left):
- `APP_URL` — e.g. `https://queuemind.vercel.app`
- `RUNTIME_SECRET` — same value as the app's `RUNTIME_SECRET` env var

In [ ]:
MODEL = "llama3.2:3b"   # fast on a T4; try "llama3.1:8b" for better quality
from google.colab import userdata
APP_URL = userdata.get("APP_URL").rstrip("/")
RUNTIME_SECRET = userdata.get("RUNTIME_SECRET")
print("App:", APP_URL, "| Model:", MODEL)

In [ ]:
# 1. Install and start Ollama
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, os, time
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_ORIGINS="*", OLLAMA_KEEP_ALIVE="24h")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}
!ollama list

In [ ]:
# 2. Warm up the model and check JSON output locally
import requests, json, time
t = time.time()
r = requests.post("http://localhost:11434/v1/chat/completions", json={
    "model": MODEL, "temperature": 0, "response_format": {"type": "json_object"},
    "messages": [
        {"role": "system", "content": "Classify the clinic walk-in. Return JSON {specialty, est_minutes, urgency, summary}. Specialties: General Medicine, Pediatrics."},
        {"role": "user", "content": "My son has had ear pain since last night"},
    ],
}, timeout=120)
print(f"{time.time()-t:.1f}s", r.json()["choices"][0]["message"]["content"])

In [ ]:
# 3. Free HTTPS tunnel (Cloudflare quick tunnel — no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import re, subprocess, time
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
deadline = time.time() + 60
while time.time() < deadline and not PUBLIC_URL:
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: PUBLIC_URL = m.group(0)
print("Public URL:", PUBLIC_URL)
time.sleep(5)
print(requests.get(f"{PUBLIC_URL}/api/tags", timeout=20).status_code)

In [ ]:
# 4. Register with QueueMind + heartbeat every 30s (runs in the background)
import threading
def heartbeat():
    while True:
        try:
            r = requests.post(f"{APP_URL}/api/runtime", json={
                "secret": RUNTIME_SECRET, "url": PUBLIC_URL, "model": MODEL,
                "meta": {"gpu": "T4", "host": "colab"}}, timeout=15)
            print(time.strftime("%H:%M:%S"), "heartbeat", r.status_code, end="")
        except Exception as e:
            print("heartbeat failed:", e)
        time.sleep(30)
threading.Thread(target=heartbeat, daemon=True).start()
print("Registered. Check the status pill in the app — it should say 'Self-hosted Llama: online'.")

In [ ]:
# 5. Keep the notebook alive and show the router's view (re-run any time)
print(json.dumps(requests.get(f"{APP_URL}/api/status", timeout=20).json(), indent=2))